# DRI-23 INT4 Quantization With AutoGPTQ

This notebook merges the selected run #4 LoRA checkpoint into `Qwen/Qwen2-VL-7B-Instruct`, quantizes the merged model to INT4 with the Qwen2-VL AutoGPTQ path, evaluates the quantized model on TBX11K val, and uploads artifacts to Hugging Face.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "codex/dri23-gptq-quantization"  # change to main after the PR lands
WORKDIR = Path("/content/Drishti")

MERGED_DIR = Path("outputs/dri23-run4-merged-fp16")
QUANT_DIR = Path("outputs/dri23-run4-gptq-int4")
EVAL_DIR = Path("outputs/eval/dri23-run4-gptq-int4")
HF_QUANT_REPO_ID = "ShivSingh123/drishti-qwen2vl-run4-gptq-int4"

BASE_MODEL = "Qwen/Qwen2-VL-7B-Instruct"
ADAPTER_REPO_ID = "ShivSingh123/drishti-qlora-run4-vision-lora-ablation"
ADAPTER_REPO_PATH = "checkpoints/checkpoint-4950"
print(BRANCH, HF_QUANT_REPO_ID)

In [ ]:
import os, subprocess, sys

def run(command: list[str], *, cwd: Path | None = None) -> None:
    print("\n$ " + " ".join(command))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)

if WORKDIR.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=WORKDIR)
    run(["git", "checkout", BRANCH], cwd=WORKDIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=WORKDIR)
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(WORKDIR)])

os.chdir(WORKDIR)
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"])
run([sys.executable, "-m", "pip", "install", "qwen-vl-utils"])
run([sys.executable, "-m", "pip", "install", "git+https://github.com/kq-chen/AutoGPTQ.git"])
print(Path.cwd())

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
KAGGLE_KEY = userdata.get("KAGGLE_KEY")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")

if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN to Colab secrets before running DRI-23.")
if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError("Add KAGGLE_USERNAME and KAGGLE_KEY to Colab secrets before running DRI-23.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_QUANT_REPO_ID, repo_type="model", private=True, exist_ok=True)
print("Secrets ready; HF repo exists.")

In [ ]:
!nvidia-smi
import importlib.metadata as md
import torch

print("torch", torch.__version__)
print("cuda", torch.version.cuda)
print("gpu available", torch.cuda.is_available())
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
for package in ["transformers", "peft", "accelerate", "auto-gptq", "qwen-vl-utils", "huggingface_hub"]:
    try:
        print(package, md.version(package))
    except md.PackageNotFoundError:
        print(package, "not installed")

if not torch.cuda.is_available():
    raise RuntimeError("DRI-23 quantization requires a CUDA GPU runtime.")

In [ ]:
run(["python", "download_dataset.py"])
run(["python", "generate_jsonl.py", "--output-dir", "data/processed"])
run(["python", "-m", "unittest", "-v", "test_quantization_utils.py"])
run(["python", "merge_lora_checkpoint.py", "--help"])
run(["python", "quantize_autogptq_qwen2vl.py", "--help"])
run(["python", "evaluate_quantized_checkpoint.py", "--help"])

## Merge LoRA Into The Base Model

This produces the full-precision merged model that GPTQ quantizes. Expect this to use much more disk than the final INT4 output.

In [ ]:
run([
    "python", "merge_lora_checkpoint.py",
    "--base-model", BASE_MODEL,
    "--adapter-repo-id", ADAPTER_REPO_ID,
    "--adapter-repo-path", ADAPTER_REPO_PATH,
    "--output-dir", str(MERGED_DIR),
    "--torch-dtype", "bfloat16",
])

## Quantize With AutoGPTQ

Calibration uses 128 stratified TBX11K val samples. If this OOMs, restart the runtime and retry with a smaller `--batch-size 1` is already the default; the next real fallback is an L4/A100 runtime with more memory.

In [ ]:
run([
    "python", "quantize_autogptq_qwen2vl.py",
    "--merged-model-dir", str(MERGED_DIR),
    "--data-dir", "data/processed",
    "--split", "val",
    "--calibration-samples", "128",
    "--output-dir", str(QUANT_DIR),
    "--bits", "4",
    "--group-size", "128",
    "--batch-size", "1",
    "--size-limit-gb", "4.0",
])

## Evaluate Quantized Model

Run a small smoke eval first, then full validation. The full eval attaches the delta vs the run #4 full-precision baseline and fails if macro-F1 drops by more than 0.02.

In [ ]:
run([
    "python", "evaluate_quantized_checkpoint.py",
    "--model-dir", str(QUANT_DIR),
    "--data-dir", "data/processed",
    "--split", "val",
    "--output-dir", str(EVAL_DIR / "smoke-30"),
    "--batch-size", "1",
    "--limit", "30",
    "--generation-smoke",
])

run([
    "python", "evaluate_quantized_checkpoint.py",
    "--model-dir", str(QUANT_DIR),
    "--data-dir", "data/processed",
    "--split", "val",
    "--output-dir", str(EVAL_DIR / "full-val"),
    "--batch-size", "3",
    "--gate", "run3-full",
    "--generation-smoke",
    "--fail-on-gate-fail",
])

In [ ]:
import json

metrics = json.loads((EVAL_DIR / "full-val" / "eval_results.json").read_text())
print(json.dumps({
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "prediction_distribution": metrics["prediction_distribution"],
    "quantization_delta": metrics["quantization_delta"],
    "structured_output_smoke": metrics.get("structured_output_smoke"),
    "gate": metrics.get("gate"),
}, indent=2))

In [ ]:
api.upload_folder(
    repo_id=HF_QUANT_REPO_ID,
    repo_type="model",
    folder_path=str(QUANT_DIR),
    path_in_repo="gptq-int4",
)
api.upload_folder(
    repo_id=HF_QUANT_REPO_ID,
    repo_type="model",
    folder_path=str(EVAL_DIR),
    path_in_repo="eval",
)
print(f"Uploaded quantized model and eval artifacts to https://huggingface.co/{HF_QUANT_REPO_ID}")